# 03 - Evaluation & Clinical Safety Analysis

This notebook loads the trained results and produces:
- AUC-ROC comparison table
- ROC curves per pathology overlaid by strategy
- Grouped bar charts (AUC-ROC, AUC-PR, Sensitivity @ 90% Specificity)
- Clinical safety discussion

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab.')

In [ ]:
import os
import sys
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/Research Project'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_config
from src.evaluate import (
    build_summary_table,
    compute_aurocs,
    compute_auprcs,
    plot_auc_comparison,
    plot_roc_curves,
    plot_sensitivity_comparison,
    sensitivity_at_specificity,
)

config = load_config(os.path.join(PROJECT_ROOT, 'configs', 'default.yaml'))
TARGET_LABELS = config['data']['target_labels']

In [ ]:
CKPT_DIR = config['checkpointing']['save_dir']
results_path = os.path.join(CKPT_DIR, 'all_results.pkl')

with open(results_path, 'rb') as f:
    all_results = pickle.load(f)

print(f'Loaded results for {len(all_results)} strategies:')
for r in all_results:
    print(f"  {r['strategy']:20s}  mean AUC = {r['best_auc']:.4f}")

## 1. AUC-ROC Summary Table

In [ ]:
from IPython.display import Markdown

table_md = build_summary_table(all_results, TARGET_LABELS)
display(Markdown(table_md))

## 2. AUC-PR per Strategy

In [ ]:
for r in all_results:
    r['auprcs'] = compute_auprcs(r['labels'], r['preds'], TARGET_LABELS)

rows = []
for r in all_results:
    row = {'Strategy': r['strategy']}
    for name in TARGET_LABELS:
        row[name] = r['auprcs'].get(name, float('nan'))
    row['Mean'] = np.nanmean(list(r['auprcs'].values()))
    rows.append(row)

df_auprc = pd.DataFrame(rows).set_index('Strategy')
df_auprc.style.format('{:.3f}').highlight_max(axis=0)

## 3. ROC Curves

In [ ]:
fig = plot_roc_curves(all_results, TARGET_LABELS, figsize=(20, 4))
plt.show()

## 4. AUC-ROC Grouped Bar Chart

In [ ]:
fig = plot_auc_comparison(all_results, TARGET_LABELS)
plt.show()

## 5. Clinical Safety: Sensitivity @ 90% Specificity

A clinically *safe* model should maximise sensitivity (not miss true positives) at a high specificity operating point. This is especially important for diseases where a missed diagnosis is dangerous.

In [ ]:
fig = plot_sensitivity_comparison(all_results, TARGET_LABELS, target_specificity=0.90)
plt.show()

In [ ]:
rows = []
for r in all_results:
    sens = sensitivity_at_specificity(r['labels'], r['preds'], 0.90, TARGET_LABELS)
    row = {'Strategy': r['strategy']}
    for name in TARGET_LABELS:
        row[name] = sens.get(name, float('nan'))
    row['Mean'] = np.nanmean(list(sens.values()))
    rows.append(row)

df_sens = pd.DataFrame(rows).set_index('Strategy')
df_sens.style.format('{:.3f}').highlight_max(axis=0)

## 6. Best Strategy per Pathology

In [ ]:
rows_auc = []
for r in all_results:
    row = {'Strategy': r['strategy']}
    for name in TARGET_LABELS:
        row[name] = r['aurocs'].get(name, float('nan'))
    rows_auc.append(row)

df_auc = pd.DataFrame(rows_auc).set_index('Strategy')

print('Best strategy per pathology (by AUC-ROC):')
for col in TARGET_LABELS:
    best = df_auc[col].idxmax()
    val = df_auc[col].max()
    print(f'  {col:25s}: {best:20s} (AUC = {val:.3f})')

## 7. Discussion & Clinical Implications

**Key questions for your paper:**

1. Which strategy yields the highest overall AUC? Does a single strategy dominate, or is the best strategy pathology-dependent (as the original CheXpert paper found)?

2. Which strategy is **safest** -- i.e., has the highest sensitivity at 90% specificity? A model that misses fewer true positives is preferable in a screening context.

3. Does **Label Smoothing** (your novel addition) outperform the original CheXpert strategies for any pathology? What trade-offs does it introduce?

4. For pathologies with high uncertain-label rates (e.g. Atelectasis), do strategies that *include* uncertain labels (U-Ones, Label Smoothing) outperform those that *exclude* them (U-Ignore)?

5. Is there a tension between AUC-ROC and clinical safety (sensitivity)? A model with slightly lower AUC but higher sensitivity at a fixed specificity may be more clinically valuable.